# CLIP-Style Contrastive Fine-Tuning (ViT + Small Text Encoder)

This notebook shows a simple, demonstrative pipeline to fine-tune:
- Visual encoder: **ViT-Base** (`google/vit-base-patch16-224-in21k`)
- Text encoder: **small open-source encoder** (`prajjwal1/bert-tiny`)

assume paired image-text data is already prepared.

## Expected Dataset Format

Use paired samples like:
```python
{
  "image_path": "/path/to/image.jpg",
  "text": "a short caption for the image"
}
```

Minimum requirements:
- `image_path`: path to an RGB image
- `text`: one caption/string per image

You can load this from JSON/JSONL/CSV/Parquet as long as you produce the same fields.

In [ ]:
# If needed:
# !pip install -q torch torchvision transformers pillow

import math
from dataclasses import dataclass
from typing import List, Dict

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from PIL import Image

from transformers import (
    AutoModel,
    AutoTokenizer,
    AutoImageProcessor,
    ViTModel,
)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
device

In [ ]:
# -----------------------------
# 1) Configuration
# -----------------------------
vision_model_name = "google/vit-base-patch16-224-in21k"
text_model_name = "prajjwal1/bert-tiny"

projection_dim = 256
max_text_len = 32
batch_size = 32
num_epochs = 3
learning_rate = 3e-5
weight_decay = 1e-4

image_processor = AutoImageProcessor.from_pretrained(vision_model_name)
tokenizer = AutoTokenizer.from_pretrained(text_model_name)

In [ ]:
# -----------------------------
# 2) Pseudo data loading
# -----------------------------
# Replace this with your real loading code.
# Example expected objects after loading:
# train_samples = [
#   {"image_path": "/data/train/0001.jpg", "text": "a dog running in the park"},
#   {"image_path": "/data/train/0002.jpg", "text": "a close-up of a red flower"},
# ]
# val_samples = [ ... ]

train_samples = []  # TODO: fill with your prepared data
val_samples = []    # TODO: fill with your prepared data

class ImageTextDataset(Dataset):
    def __init__(self, samples: List[Dict]):
        self.samples = samples

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        image = Image.open(item["image_path"]).convert("RGB")
        text = item["text"]
        return {"image": image, "text": text}

def collate_fn(batch):
    # Image preprocessing from Hugging Face image processor
    images = [x["image"] for x in batch]
    image_inputs = image_processor(images=images, return_tensors="pt")

    # Text tokenization from Hugging Face tokenizer
    texts = [x["text"] for x in batch]
    text_inputs = tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_text_len,
        return_tensors="pt",
    )

    return {
        "pixel_values": image_inputs["pixel_values"],
        "input_ids": text_inputs["input_ids"],
        "attention_mask": text_inputs["attention_mask"],
    }

train_dataset = ImageTextDataset(train_samples)
val_dataset = ImageTextDataset(val_samples)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, collate_fn=collate_fn)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, collate_fn=collate_fn)

print(f"Train samples: {len(train_dataset)} | Val samples: {len(val_dataset)}")

In [ ]:
# -----------------------------
# 3) CLIP-style model
# -----------------------------
class CLIPLikeModel(nn.Module):
    def __init__(self, vision_name: str, text_name: str, proj_dim: int = 256):
        super().__init__()

        # Vision encoder (ViT-Base)
        self.vision_encoder = ViTModel.from_pretrained(vision_name)

        # Text encoder (small open-source model)
        self.text_encoder = AutoModel.from_pretrained(text_name)

        vision_hidden = self.vision_encoder.config.hidden_size
        text_hidden = self.text_encoder.config.hidden_size

        # Projection heads map both modalities to a shared embedding space
        self.vision_proj = nn.Linear(vision_hidden, proj_dim)
        self.text_proj = nn.Linear(text_hidden, proj_dim)

        # Learnable temperature parameter (as in CLIP)
        self.logit_scale = nn.Parameter(torch.tensor(math.log(1 / 0.07)))

    def encode_image(self, pixel_values):
        outputs = self.vision_encoder(pixel_values=pixel_values)
        cls_embed = outputs.last_hidden_state[:, 0]  # [CLS]
        image_embeds = self.vision_proj(cls_embed)
        return F.normalize(image_embeds, dim=-1)

    def encode_text(self, input_ids, attention_mask):
        outputs = self.text_encoder(input_ids=input_ids, attention_mask=attention_mask)
        cls_embed = outputs.last_hidden_state[:, 0]  # [CLS]
        text_embeds = self.text_proj(cls_embed)
        return F.normalize(text_embeds, dim=-1)

    def forward(self, pixel_values, input_ids, attention_mask):
        image_embeds = self.encode_image(pixel_values)
        text_embeds = self.encode_text(input_ids, attention_mask)

        # Similarity matrix for all image-text pairs in the batch
        logit_scale = self.logit_scale.exp()
        logits_per_image = logit_scale * image_embeds @ text_embeds.t()
        logits_per_text = logits_per_image.t()

        return logits_per_image, logits_per_text

def clip_contrastive_loss(logits_per_image, logits_per_text):
    # Positive pairs are on the diagonal: image i <-> text i
    targets = torch.arange(logits_per_image.size(0), device=logits_per_image.device)
    loss_i = F.cross_entropy(logits_per_image, targets)
    loss_t = F.cross_entropy(logits_per_text, targets)
    return (loss_i + loss_t) / 2

model = CLIPLikeModel(vision_model_name, text_model_name, proj_dim=projection_dim).to(device)
model

In [ ]:
# -----------------------------
# 4) Optimizer
# -----------------------------
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=weight_decay)

In [ ]:
# -----------------------------
# 5) Train / eval loops
# -----------------------------
def train_one_epoch(model, dataloader, optimizer, device):
    model.train()
    running_loss = 0.0

    for batch in dataloader:
        pixel_values = batch["pixel_values"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        logits_per_image, logits_per_text = model(pixel_values, input_ids, attention_mask)
        loss = clip_contrastive_loss(logits_per_image, logits_per_text)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    return running_loss / max(len(dataloader), 1)

@torch.no_grad()
def evaluate(model, dataloader, device):
    model.eval()
    running_loss = 0.0

    for batch in dataloader:
        pixel_values = batch["pixel_values"].to(device)
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)

        logits_per_image, logits_per_text = model(pixel_values, input_ids, attention_mask)
        loss = clip_contrastive_loss(logits_per_image, logits_per_text)
        running_loss += loss.item()

    return running_loss / max(len(dataloader), 1)

In [ ]:
# -----------------------------
# 6) Run training
# -----------------------------
if len(train_dataset) == 0:
    print("Please load your data into train_samples/val_samples first.")
else:
    for epoch in range(1, num_epochs + 1):
        train_loss = train_one_epoch(model, train_loader, optimizer, device)
        val_loss = evaluate(model, val_loader, device) if len(val_dataset) > 0 else float('nan')
        print(f"Epoch {epoch}/{num_epochs} | train_loss={train_loss:.4f} | val_loss={val_loss:.4f}")

    # Save model weights
    torch.save(model.state_dict(), "clip_like_vit_bert_tiny.pt")
    print("Saved: clip_like_vit_bert_tiny.pt")